# Visualisation par point de données — `blob_nn_4x10-1`

Ce notebook affiche **chaque point de données individuellement** (pas de moyenne par
combo) : un point = une ligne du CSV = un `(combo, data_index)`.

**Colonnes attendues** :
`combo, strategy, l, u_idx, k, j_idx, data_index, target, certified, optimal_value, gain, LB_neuron1, UB_neuron1, LB_neuron2, UB_neuron2`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# === CONFIGURATION ===
INPUT_CSV = "combo_detail_by_datapoint.csv"

COLUMNS = [
    "combo", "strategy", "l", "u_idx", "k", "j_idx",
    "data_index", "target", "certified", "optimal_value", "gain",
    "LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2",
]

NUMERIC_COLS = [
    "l", "u_idx", "k", "j_idx", "data_index", "target",
    "optimal_value", "gain", "LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2",
]

# Détection automatique de la présence d'un en-tête.
with open(INPUT_CSV) as f:
    first_line = f.readline().strip().split(",")

has_header = first_line[:2] == ["combo", "strategy"]

if has_header:
    df = pd.read_csv(INPUT_CSV)
    missing = set(COLUMNS) - set(df.columns)
    if missing:
        raise ValueError(f"Colonnes attendues manquantes dans {INPUT_CSV}: {missing}")
else:
    df = pd.read_csv(INPUT_CSV, header=None, names=COLUMNS)

for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["certified"] = df["certified"].astype(str).str.lower().isin(["true", "1"])

print(f"En-tête détecté : {has_header}")
print(f"{len(df)} lignes (points de données), {df['combo'].nunique()} combo(s)")
df.head()


## 1. Gain par point de données, dans l'ordre du fichier\n\nChaque point = une ligne. Couleur = certifié ou non.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

colors = np.where(df["certified"], "#2a9d8f", "#e76f51")
ax.scatter(range(len(df)), df["gain"], c=colors, alpha=0.5, s=10)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Index de ligne (ordre du fichier)")
ax.set_ylabel("Gain")
ax.set_title("Gain par point de données")

handles = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#2a9d8f", label="certifié", markersize=6),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#e76f51", label="non certifié", markersize=6),
]
ax.legend(handles=handles)
plt.tight_layout()
plt.show()


## 2. Gain par point de données, par combo (nuage de points, un combo par ligne)\n\nChaque combo occupe une bande horizontale ; chaque point y est dispersé verticalement (jitter) juste pour éviter la superposition — la position verticale dans la bande n'a pas de sens en elle-même, seule la couleur/l'abscisse (gain) compte.

In [ ]:
# Pour la lisibilité, on peut limiter le nombre de combos affichés.
N_MAX_COMBOS = 40
combos = df["combo"].dropna().unique()
if len(combos) > N_MAX_COMBOS:
    print(f"[!] {len(combos)} combos détectés, affichage limité aux {N_MAX_COMBOS} premiers (triés par nom). "
          f"Changez N_MAX_COMBOS pour en voir plus.")
    combos = sorted(combos)[:N_MAX_COMBOS]
else:
    combos = sorted(combos)

subset = df[df["combo"].isin(combos)].copy()
combo_to_y = {c: i for i, c in enumerate(combos)}
subset["y"] = subset["combo"].map(combo_to_y) + np.random.uniform(-0.3, 0.3, size=len(subset))

fig, ax = plt.subplots(figsize=(12, max(4, 0.25 * len(combos))))
colors = np.where(subset["certified"], "#2a9d8f", "#e76f51")
ax.scatter(subset["gain"], subset["y"], c=colors, alpha=0.5, s=10)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(range(len(combos)))
ax.set_yticklabels(combos)
ax.set_xlabel("Gain")
ax.set_title("Gain — chaque point individuel, par combo")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 3. Gain vs largeur des bornes des neurones (LB/UB)\n\nChaque point = un `(combo, data_index)`.

In [ ]:
plot_df = df.dropna(subset=["gain", "LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2"]).copy()
plot_df["width_neuron1"] = plot_df["UB_neuron1"] - plot_df["LB_neuron1"]
plot_df["width_neuron2"] = plot_df["UB_neuron2"] - plot_df["LB_neuron2"]
plot_df["width_product"] = plot_df["width_neuron1"] * plot_df["width_neuron2"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(plot_df["width_neuron1"], plot_df["gain"], alpha=0.35, s=10, label="neurone 1")
axes[0].scatter(plot_df["width_neuron2"], plot_df["gain"], alpha=0.35, s=10, label="neurone 2")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("Largeur de l'intervalle [LB, UB]")
axes[0].set_ylabel("Gain")
axes[0].set_title("Gain vs largeur d'intervalle (par neurone)")
axes[0].legend()

axes[1].scatter(plot_df["width_product"], plot_df["gain"], alpha=0.35, s=10, color="#e76f51")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("Largeur neurone1 × Largeur neurone2")
axes[1].set_ylabel("Gain")
axes[1].set_title("Gain vs produit des largeurs")

plt.tight_layout()
plt.show()

corr1 = plot_df["width_neuron1"].corr(plot_df["gain"])
corr2 = plot_df["width_neuron2"].corr(plot_df["gain"])
corr_prod = plot_df["width_product"].corr(plot_df["gain"])
print(f"Corrélation gain / largeur neurone1 : {corr1:.3f}")
print(f"Corrélation gain / largeur neurone2 : {corr2:.3f}")
print(f"Corrélation gain / largeur produit  : {corr_prod:.3f}")


## 4. Gain vs data_index (position dans le test set), tous combos superposés

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
colors = np.where(df["certified"], "#2a9d8f", "#e76f51")
ax.scatter(df["data_index"], df["gain"], c=colors, alpha=0.4, s=10)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("data_index (position du point testé)")
ax.set_ylabel("Gain")
ax.set_title("Gain vs data_index — tous les combos, chaque point individuel")
plt.tight_layout()
plt.show()


## 5. optimal_value par point de données (combo vs baseline)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
colors = np.where(df["certified"], "#2a9d8f", "#e76f51")
ax.scatter(range(len(df)), df["optimal_value"], c=colors, alpha=0.5, s=10)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Index de ligne (ordre du fichier)")
ax.set_ylabel("optimal_value")
ax.set_title("optimal_value par point de données (certifié = optimal_value > 0)")
plt.tight_layout()
plt.show()


## 6. Table brute triable/filtrable

In [ ]:
# Exemple : les points où le gain est le plus négatif (la stratégie fait moins bien que le baseline)
df.sort_values("gain").head(30)


## 7. Découverte de patterns (arbre de décision)

On entraîne un **arbre de décision** pour trouver des règles simples ("si largeur du
neurone 1 > X et l = 2 alors gain élevé") qui expliquent le `gain` et la
`certification`, à partir de features disponibles pour chaque point de données :
`l, u_idx, k, j_idx, LB/UB des 2 neurones, largeur des intervalles, data_index`.

**Attention à l'interprétation** : un arbre profond peut sur-apprendre (mémoriser le
bruit plutôt que capturer un vrai pattern). On limite volontairement la profondeur
(`max_depth`) pour ne garder que des règles robustes et lisibles — augmentez-la
prudemment si besoin, en gardant un œil sur le score de validation.


In [ ]:
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, accuracy_score

# === Préparation des features ===
tree_df = df.dropna(subset=["gain", "LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2"]).copy()
tree_df["width_neuron1"] = tree_df["UB_neuron1"] - tree_df["LB_neuron1"]
tree_df["width_neuron2"] = tree_df["UB_neuron2"] - tree_df["LB_neuron2"]
tree_df["width_product"] = tree_df["width_neuron1"] * tree_df["width_neuron2"]

FEATURE_COLS = [
    "l", "u_idx", "k", "j_idx",
    "LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2",
    "width_neuron1", "width_neuron2", "width_product",
    "data_index",
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in tree_df.columns]

X = tree_df[FEATURE_COLS].fillna(tree_df[FEATURE_COLS].median())
y_gain = tree_df["gain"]
y_cert = tree_df["certified"].astype(int)

print(f"{len(tree_df)} points de données utilisés, {len(FEATURE_COLS)} features : {FEATURE_COLS}")


### 7a. Arbre de régression — LB/UB seuls, puis avec `k` et `l`

Même logique que pour la classification : on regarde si `k` et `l` ajoutent du
pouvoir prédictif sur le `gain` au-delà des seules bornes des 2 neurones.
- **Modèle 1** : `LB_neuron1, UB_neuron1, LB_neuron2, UB_neuron2` uniquement.
- **Modèle 2** : Modèle 1 + `k, l`.


In [ ]:
MAX_DEPTH_REG = 3  # augmentez prudemment si besoin de règles plus fines

def fit_and_report_reg_tree(feature_cols, title):
    Xr = tree_df[feature_cols].fillna(tree_df[feature_cols].median())
    yr = tree_df["gain"]

    X_train, X_test, y_train, y_test = train_test_split(Xr, yr, test_size=0.25, random_state=0)

    reg = DecisionTreeRegressor(max_depth=MAX_DEPTH_REG, min_samples_leaf=10, random_state=0)
    reg.fit(X_train, y_train)

    r2_train = r2_score(y_train, reg.predict(X_train))
    r2_test = r2_score(y_test, reg.predict(X_test))

    print(f"--- {title} ---")
    print(f"Features: {feature_cols}")
    print(f"R² train: {r2_train:.3f} | R² test: {r2_test:.3f}")
    if r2_test < 0.1:
        print("[!] R² test faible : l'arbre n'explique presque rien du gain avec ces features.")

    fig, ax = plt.subplots(figsize=(20, 10))
    plot_tree(reg, feature_names=feature_cols, filled=True, rounded=True, fontsize=9, ax=ax, precision=2)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

    print("\nRègles extraites :\n")
    print(export_text(reg, feature_names=feature_cols))
    print()

    return reg, r2_test


FEATURES_BOUNDS_ONLY = ["LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2"]
FEATURES_BOUNDS_PLUS_KL = FEATURES_BOUNDS_ONLY + ["k", "l"]

reg_bounds, r2_bounds = fit_and_report_reg_tree(
    FEATURES_BOUNDS_ONLY, "Modèle 1 — LB/UB des 2 neurones uniquement"
)
reg_bounds_kl, r2_bounds_kl = fit_and_report_reg_tree(
    FEATURES_BOUNDS_PLUS_KL, "Modèle 2 — LB/UB + k, l"
)

print(f"Gain de R² en ajoutant k, l : {r2_bounds_kl - r2_bounds:+.3f}")


### 7b. Arbre de classification — LB/UB seuls, puis avec `k` et `l`

Deux modèles pour voir si la position du produit croisé (`l`, `k`) ajoute du pouvoir
prédictif au-delà des seules bornes des 2 neurones :
- **Modèle 1** : `LB_neuron1, UB_neuron1, LB_neuron2, UB_neuron2` uniquement.
- **Modèle 2** : Modèle 1 + `k, l`.


In [ ]:
MAX_DEPTH_CLF = 3

def fit_and_report_tree(feature_cols, title):
    Xc = tree_df[feature_cols].fillna(tree_df[feature_cols].median())
    yc = tree_df["certified"].astype(int)

    X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
        Xc, yc, test_size=0.25, random_state=0, stratify=yc
    )

    clf = DecisionTreeClassifier(max_depth=MAX_DEPTH_CLF, min_samples_leaf=10, random_state=0)
    clf.fit(X_train_c, y_train_c)

    acc_train = accuracy_score(y_train_c, clf.predict(X_train_c))
    acc_test = accuracy_score(y_test_c, clf.predict(X_test_c))
    baseline_acc = max(yc.mean(), 1 - yc.mean())

    print(f"--- {title} ---")
    print(f"Features: {feature_cols}")
    print(f"Accuracy train: {acc_train:.3f} | Accuracy test: {acc_test:.3f} | Baseline (classe majoritaire): {baseline_acc:.3f}")
    if acc_test <= baseline_acc + 0.02:
        print("[!] L'arbre ne fait pas mieux que la classe majoritaire avec ces features.")

    fig, ax = plt.subplots(figsize=(20, 10))
    plot_tree(
        clf, feature_names=feature_cols, class_names=["non certifié", "certifié"],
        filled=True, rounded=True, fontsize=9, ax=ax,
    )
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

    print("\nRègles extraites :\n")
    print(export_text(clf, feature_names=feature_cols))
    print()

    return clf, acc_test


FEATURES_BOUNDS_ONLY = ["LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2"]
FEATURES_BOUNDS_PLUS_KL = FEATURES_BOUNDS_ONLY + ["k", "l"]

clf_bounds, acc_bounds = fit_and_report_tree(
    FEATURES_BOUNDS_ONLY, "Modèle 1 — LB/UB des 2 neurones uniquement"
)
clf_bounds_kl, acc_bounds_kl = fit_and_report_tree(
    FEATURES_BOUNDS_PLUS_KL, "Modèle 2 — LB/UB + k, l"
)

print(f"Gain d'accuracy en ajoutant k, l : {acc_bounds_kl - acc_bounds:+.3f}")


### 7c. Importance des features (les deux arbres)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

imp_reg_bounds = pd.Series(reg_bounds.feature_importances_, index=FEATURES_BOUNDS_ONLY).sort_values()
axes[0, 0].barh(imp_reg_bounds.index, imp_reg_bounds.values, color="#2a9d8f")
axes[0, 0].set_title("Régression (gain) — LB/UB seuls")

imp_reg_kl = pd.Series(reg_bounds_kl.feature_importances_, index=FEATURES_BOUNDS_PLUS_KL).sort_values()
axes[0, 1].barh(imp_reg_kl.index, imp_reg_kl.values, color="#2a9d8f")
axes[0, 1].set_title("Régression (gain) — LB/UB + k, l")

imp_bounds = pd.Series(clf_bounds.feature_importances_, index=FEATURES_BOUNDS_ONLY).sort_values()
axes[1, 0].barh(imp_bounds.index, imp_bounds.values, color="#264653")
axes[1, 0].set_title("Classification — LB/UB seuls")

imp_bounds_kl = pd.Series(clf_bounds_kl.feature_importances_, index=FEATURES_BOUNDS_PLUS_KL).sort_values()
axes[1, 1].barh(imp_bounds_kl.index, imp_bounds_kl.values, color="#e76f51")
axes[1, 1].set_title("Classification — LB/UB + k, l")

plt.tight_layout()
plt.show()


## 8. Frontières de décision de l'arbre, avec les points réels

Un arbre à plusieurs features ne se visualise pas directement en 2D. Pour voir
concrètement **où l'arbre trace ses coupures par rapport aux points**, on reprend,
pour chaque modèle, ses **2 features les plus importantes** et on ré-entraîne un
arbre 2D (même profondeur) juste pour l'affichage : le fond coloré montre la
prédiction de l'arbre sur la grille, les points sont les vraies données.

⚠️ C'est une **vue simplifiée** : l'arbre 2D ré-entraîné sur seulement 2 features
peut différer légèrement de l'arbre complet (3+ features) présenté en section 7 —
il sert à *illustrer* la logique de coupure, pas à remplacer l'arbre complet.


In [ ]:
def plot_2d_decision_regions(X_full, y, feature_names, is_classifier, max_depth, title):
    """Ré-entraîne un arbre sur exactement 2 features et affiche ses régions de
    décision (fond) superposées aux points réels."""
    f1, f2 = feature_names
    X2 = X_full[[f1, f2]]

    if is_classifier:
        model2d = DecisionTreeClassifier(max_depth=max_depth, min_samples_leaf=10, random_state=0)
    else:
        model2d = DecisionTreeRegressor(max_depth=max_depth, min_samples_leaf=10, random_state=0)
    model2d.fit(X2, y)

    pad1 = (X2[f1].max() - X2[f1].min()) * 0.05 or 1
    pad2 = (X2[f2].max() - X2[f2].min()) * 0.05 or 1
    xx, yy = np.meshgrid(
        np.linspace(X2[f1].min() - pad1, X2[f1].max() + pad1, 300),
        np.linspace(X2[f2].min() - pad2, X2[f2].max() + pad2, 300),
    )
    grid = pd.DataFrame({f1: xx.ravel(), f2: yy.ravel()})
    zz = model2d.predict(grid).reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(8, 6))

    if is_classifier:
        ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5], colors=["#fbe3dd", "#d7efe9"], alpha=0.8)
        colors = np.where(y == 1, "#2a9d8f", "#e76f51")
        ax.scatter(X2[f1], X2[f2], c=colors, s=12, alpha=0.7, edgecolor="none")
        handles = [
            plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#2a9d8f", label="certifié", markersize=7),
            plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#e76f51", label="non certifié", markersize=7),
        ]
        ax.legend(handles=handles)
    else:
        vmax = np.nanmax(np.abs(zz)) or 1
        cf = ax.contourf(xx, yy, zz, levels=20, cmap="RdYlGn", vmin=-vmax, vmax=vmax, alpha=0.75)
        fig.colorbar(cf, ax=ax, label="gain prédit (arbre 2D)")
        ax.scatter(X2[f1], X2[f2], c=y, cmap="RdYlGn", vmin=-vmax, vmax=vmax, s=14, edgecolor="black", linewidth=0.3)

    ax.set_xlabel(f1)
    ax.set_ylabel(f2)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

    return model2d


### 8a. Régression (gain) — modèle LB/UB seuls

In [ ]:
top2_reg_bounds = imp_reg_bounds.sort_values(ascending=False).index[:2].tolist()
X_reg_bounds = tree_df[FEATURES_BOUNDS_ONLY].fillna(tree_df[FEATURES_BOUNDS_ONLY].median())
_ = plot_2d_decision_regions(
    X_reg_bounds, tree_df["gain"], top2_reg_bounds, is_classifier=False,
    max_depth=MAX_DEPTH_REG, title=f"Gain — arbre 2D sur {top2_reg_bounds}",
)


### 8b. Régression (gain) — modèle LB/UB + k, l

In [ ]:
top2_reg_kl = imp_reg_kl.sort_values(ascending=False).index[:2].tolist()
X_reg_kl = tree_df[FEATURES_BOUNDS_PLUS_KL].fillna(tree_df[FEATURES_BOUNDS_PLUS_KL].median())
_ = plot_2d_decision_regions(
    X_reg_kl, tree_df["gain"], top2_reg_kl, is_classifier=False,
    max_depth=MAX_DEPTH_REG, title=f"Gain — arbre 2D sur {top2_reg_kl}",
)


### 8c. Classification (certified) — modèle LB/UB seuls

In [ ]:
top2_clf_bounds = imp_bounds.sort_values(ascending=False).index[:2].tolist()
X_clf_bounds = tree_df[FEATURES_BOUNDS_ONLY].fillna(tree_df[FEATURES_BOUNDS_ONLY].median())
y_clf = tree_df["certified"].astype(int)
_ = plot_2d_decision_regions(
    X_clf_bounds, y_clf, top2_clf_bounds, is_classifier=True,
    max_depth=MAX_DEPTH_CLF, title=f"Certification — arbre 2D sur {top2_clf_bounds}",
)


### 8d. Classification (certified) — modèle LB/UB + k, l

In [ ]:
top2_clf_kl = imp_bounds_kl.sort_values(ascending=False).index[:2].tolist()
X_clf_kl = tree_df[FEATURES_BOUNDS_PLUS_KL].fillna(tree_df[FEATURES_BOUNDS_PLUS_KL].median())
_ = plot_2d_decision_regions(
    X_clf_kl, y_clf, top2_clf_kl, is_classifier=True,
    max_depth=MAX_DEPTH_CLF, title=f"Certification — arbre 2D sur {top2_clf_kl}",
)


## 9. Au-delà de l'arbre : termes quadratiques et noyau RBF

L'arbre de décision ne capture que des coupures rectangulaires (axis-aligned). On
essaie ici deux familles de modèles capables de capturer des frontières courbes :

- **Polynomial (degré 2)** : on ajoute les termes `u²`, `l²`, `u·l`, etc. (toutes les
  combinaisons quadratiques des features) à une régression logistique (classification)
  ou une régression Ridge (gain), pour voir si une frontière quadratique explique
  mieux les données qu'une frontière en escalier.
- **RBF (noyau gaussien)** : `SVC(kernel="rbf")` pour la classification et
  `SVR(kernel="rbf")` pour le gain — capture des frontières lisses et arbitrairement
  courbes, sans forme paramétrique imposée.

On compare toujours **Modèle 1 (LB/UB seuls)** et **Modèle 2 (LB/UB + k, l)**, avec
les mêmes splits train/test que pour l'arbre (pour rester comparable).


In [ ]:
from sklearn.svm import SVC, SVR
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import make_pipeline

RANDOM_STATE = 0

def eval_classifier(model, X, y, title):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
    )
    model.fit(X_train, y_train)
    acc_train = accuracy_score(y_train, model.predict(X_train))
    acc_test = accuracy_score(y_test, model.predict(X_test))
    baseline_acc = max(y.mean(), 1 - y.mean())
    print(f"{title:45s} | acc train: {acc_train:.3f} | acc test: {acc_test:.3f} | baseline: {baseline_acc:.3f}")
    return model, acc_test


def eval_regressor(model, X, y, title):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)
    r2_train = r2_score(y_train, model.predict(X_train))
    r2_test = r2_score(y_test, model.predict(X_test))
    print(f"{title:45s} | R² train: {r2_train:.3f} | R² test: {r2_test:.3f}")
    return model, r2_test


results_summary = []  # (task, feature_set, model, score_test)


### 9a. Termes quadratiques — classification (`certified`)

In [ ]:
for name, feats in [("LB/UB seuls", FEATURES_BOUNDS_ONLY), ("LB/UB + k,l", FEATURES_BOUNDS_PLUS_KL)]:
    X_ = tree_df[feats].fillna(tree_df[feats].median())
    poly_clf = make_pipeline(
        StandardScaler(),
        PolynomialFeatures(degree=2, include_bias=False),
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    )
    _, acc = eval_classifier(poly_clf, X_, y_clf, f"Quadratique — classification — {name}")
    results_summary.append(("classification", name, "quadratique", acc))


### 9b. Termes quadratiques — régression (`gain`)

In [ ]:
for name, feats in [("LB/UB seuls", FEATURES_BOUNDS_ONLY), ("LB/UB + k,l", FEATURES_BOUNDS_PLUS_KL)]:
    X_ = tree_df[feats].fillna(tree_df[feats].median())
    poly_reg = make_pipeline(
        StandardScaler(),
        PolynomialFeatures(degree=2, include_bias=False),
        Ridge(alpha=1.0, random_state=RANDOM_STATE),
    )
    _, r2 = eval_regressor(poly_reg, X_, tree_df["gain"], f"Quadratique — régression — {name}")
    results_summary.append(("régression", name, "quadratique", r2))


### 9c. Noyau RBF — classification (`certified`)

In [ ]:
for name, feats in [("LB/UB seuls", FEATURES_BOUNDS_ONLY), ("LB/UB + k,l", FEATURES_BOUNDS_PLUS_KL)]:
    X_ = tree_df[feats].fillna(tree_df[feats].median())
    rbf_clf = make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE),
    )
    _, acc = eval_classifier(rbf_clf, X_, y_clf, f"RBF — classification — {name}")
    results_summary.append(("classification", name, "rbf", acc))


### 9d. Noyau RBF — régression (`gain`)

In [ ]:
for name, feats in [("LB/UB seuls", FEATURES_BOUNDS_ONLY), ("LB/UB + k,l", FEATURES_BOUNDS_PLUS_KL)]:
    X_ = tree_df[feats].fillna(tree_df[feats].median())
    rbf_reg = make_pipeline(
        StandardScaler(),
        SVR(kernel="rbf", C=1.0, gamma="scale"),
    )
    _, r2 = eval_regressor(rbf_reg, X_, tree_df["gain"], f"RBF — régression — {name}")
    results_summary.append(("régression", name, "rbf", r2))


### 9e. Comparaison globale : arbre vs quadratique vs RBF

In [ ]:
# On rajoute les scores de l'arbre (section 7) pour comparer sur la même échelle.
results_summary.append(("classification", "LB/UB seuls", "arbre", acc_bounds))
results_summary.append(("classification", "LB/UB + k,l", "arbre", acc_bounds_kl))
results_summary.append(("régression", "LB/UB seuls", "arbre", r2_bounds))
results_summary.append(("régression", "LB/UB + k,l", "arbre", r2_bounds_kl))

comparison_df = pd.DataFrame(results_summary, columns=["tâche", "features", "modèle", "score_test"])
comparison_df = comparison_df.sort_values(["tâche", "features", "score_test"], ascending=[True, True, False])
comparison_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, task in zip(axes, ["classification", "régression"]):
    sub = comparison_df[comparison_df["tâche"] == task]
    pivot = sub.pivot(index="features", columns="modèle", values="score_test")
    pivot.plot(kind="bar", ax=ax)
    ax.set_title(f"Score test — {task}")
    ax.set_ylabel("accuracy" if task == "classification" else "R²")
    ax.axhline(0, color="black", linewidth=0.6)
    ax.legend(title="modèle")

plt.tight_layout()
plt.show()


### 9f. Frontières de décision RBF (2D), avec les points réels\n\nMême principe que la section 8, mais avec un modèle RBF ré-entraîné sur les 2 features les plus corrélées à la cible, pour visualiser des frontières courbes.

In [ ]:
def top2_by_correlation(X, y, feats):
    corrs = X[feats].apply(lambda col: abs(np.corrcoef(col, y)[0, 1]))
    return corrs.sort_values(ascending=False).index[:2].tolist()


def plot_2d_rbf_regions(X_full, y, feature_names, is_classifier, title):
    f1, f2 = feature_names
    X2 = X_full[[f1, f2]]

    if is_classifier:
        model2d = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE))
    else:
        model2d = make_pipeline(StandardScaler(), SVR(kernel="rbf", C=1.0, gamma="scale"))
    model2d.fit(X2, y)

    pad1 = (X2[f1].max() - X2[f1].min()) * 0.05 or 1
    pad2 = (X2[f2].max() - X2[f2].min()) * 0.05 or 1
    xx, yy = np.meshgrid(
        np.linspace(X2[f1].min() - pad1, X2[f1].max() + pad1, 300),
        np.linspace(X2[f2].min() - pad2, X2[f2].max() + pad2, 300),
    )
    grid = pd.DataFrame({f1: xx.ravel(), f2: yy.ravel()})
    zz = model2d.predict(grid).reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(8, 6))
    if is_classifier:
        ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5], colors=["#fbe3dd", "#d7efe9"], alpha=0.8)
        colors = np.where(y == 1, "#2a9d8f", "#e76f51")
        ax.scatter(X2[f1], X2[f2], c=colors, s=12, alpha=0.7, edgecolor="none")
    else:
        vmax = np.nanmax(np.abs(zz)) or 1
        cf = ax.contourf(xx, yy, zz, levels=20, cmap="RdYlGn", vmin=-vmax, vmax=vmax, alpha=0.75)
        fig.colorbar(cf, ax=ax, label="gain prédit (RBF)")
        ax.scatter(X2[f1], X2[f2], c=y, cmap="RdYlGn", vmin=-vmax, vmax=vmax, s=14, edgecolor="black", linewidth=0.3)

    ax.set_xlabel(f1)
    ax.set_ylabel(f2)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return model2d


X_bounds_full = tree_df[FEATURES_BOUNDS_ONLY].fillna(tree_df[FEATURES_BOUNDS_ONLY].median())
top2_rbf_clf = top2_by_correlation(X_bounds_full, y_clf, FEATURES_BOUNDS_ONLY)
_ = plot_2d_rbf_regions(X_bounds_full, y_clf, top2_rbf_clf, is_classifier=True,
                         title=f"Certification — RBF 2D sur {top2_rbf_clf}")

top2_rbf_reg = top2_by_correlation(X_bounds_full, tree_df["gain"], FEATURES_BOUNDS_ONLY)
_ = plot_2d_rbf_regions(X_bounds_full, tree_df["gain"], top2_rbf_reg, is_classifier=False,
                         title=f"Gain — RBF 2D sur {top2_rbf_reg}")


## 10. Générer une config recommandée pour TOUS les produits du réseau

`combo_0000` (baseline) met les **330 produits** en `one_variable` ; chaque
`comboXXXX` en flip **un seul** en `composed`. Le `gain` mesuré est donc
exactement l'effet de passer *ce produit précis* en `composed`, ce qui est
justement ce que nos modèles (arbre 7a/7b, RBF 9c/9d) prédisent à partir des
bornes LB/UB (+ `k, l`).

On applique donc ces modèles à **tous les produits du réseau** (pas seulement
ceux déjà testés), en récupérant leurs bornes LB/UB moyennes depuis le
`stable_actives_study.csv` du baseline, pour décider produit par produit :
`composed` si le gain prédit est positif, sinon `one_variable`.

**Adaptez `REPO_ROOT` et `BASELINE_DIR` à votre environnement WSL.**


In [ ]:
import yaml
from pathlib import Path

# === CONFIGURATION ===
REPO_ROOT = Path("FullSDPCertification")  # clone de https://github.com/ahmedsafi2/FullSDPCertification
BASELINE_YAML = REPO_ROOT / "all_product_yamls" / "combo_0000__baseline__all_one_variable.yaml"

# Dossier du run baseline dans results/benchmark (celui utilisé plus haut comme
# combo_0000 pour calculer le gain) — sert à récupérer les bornes LB/UB moyennes
# de CHAQUE neurone, indépendamment du produit testé.
BASELINE_STABLE_CSV = Path("results/benchmark/blob_nn_4x10-1/<dossier_combo_0000>/stable_actives_study.csv")

GAIN_THRESHOLD = 0.0  # composed recommandé si gain prédit > ce seuil


In [ ]:
# === 1. Charger tous les produits du réseau depuis le YAML baseline ===
baseline_yaml_data = yaml.safe_load(BASELINE_YAML.read_text(encoding="utf-8"))
bound_strategy = baseline_yaml_data["models"][0]["bound_strategy"]

products = []
for product_name, product_cfg in bound_strategy.items():
    l, u, k, j = product_cfg["key"]
    products.append({"product_name": product_name, "l": l, "u_idx": u, "k": k, "j_idx": j})

products_df = pd.DataFrame(products)
print(f"{len(products_df)} produits trouvés dans {BASELINE_YAML.name}")
products_df.head()


In [ ]:
# === 2. Bornes LB/UB moyennes par neurone, depuis stable_actives_study.csv du baseline ===
baseline_stable_df = pd.read_csv(BASELINE_STABLE_CSV)

def mean_bounds(layer: int, neuron: int) -> tuple[float, float]:
    lb_col = f"LB_Layer_{layer}_Neuron_{neuron}"
    ub_col = f"UB_Layer_{layer}_Neuron_{neuron}"
    if lb_col not in baseline_stable_df.columns or ub_col not in baseline_stable_df.columns:
        return float("nan"), float("nan")
    return baseline_stable_df[lb_col].mean(), baseline_stable_df[ub_col].mean()


records = []
for row in products_df.itertuples():
    lb1, ub1 = mean_bounds(row.l, row.u_idx)
    lb2, ub2 = mean_bounds(row.k, row.j_idx)
    records.append({
        "product_name": row.product_name, "l": row.l, "u_idx": row.u_idx,
        "k": row.k, "j_idx": row.j_idx,
        "LB_neuron1": lb1, "UB_neuron1": ub1, "LB_neuron2": lb2, "UB_neuron2": ub2,
    })

all_products_features = pd.DataFrame(records).dropna(
    subset=["LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2"]
)
n_missing = len(products_df) - len(all_products_features)
if n_missing:
    print(f"[!] {n_missing} produit(s) sans bornes trouvées dans stable_actives_study.csv (colonnes manquantes ?)")

all_products_features.head()


In [ ]:
# === 3. Prédire le gain (arbre + RBF) pour chaque produit, avec LB/UB + k, l ===
X_all = all_products_features[FEATURES_BOUNDS_PLUS_KL]

all_products_features = all_products_features.copy()
all_products_features["gain_pred_tree"] = reg_bounds_kl.predict(X_all)
all_products_features["gain_pred_rbf"] = rbf_reg.predict(X_all) if "rbf_reg" in dir() else float("nan")

all_products_features["strategy_tree"] = np.where(
    all_products_features["gain_pred_tree"] > GAIN_THRESHOLD, "composed", "one_variable"
)
all_products_features["strategy_rbf"] = np.where(
    all_products_features["gain_pred_rbf"] > GAIN_THRESHOLD, "composed", "one_variable"
)

agreement = (all_products_features["strategy_tree"] == all_products_features["strategy_rbf"]).mean()
print(f"Accord arbre / RBF sur la stratégie recommandée : {agreement * 100:.1f}%")
print(all_products_features["strategy_tree"].value_counts().rename("arbre"))
print(all_products_features["strategy_rbf"].value_counts().rename("rbf"))

all_products_features.sort_values("gain_pred_tree", ascending=False).head(15)


### 10a. Écrire les configs YAML recommandées (une par modèle)

In [ ]:
def write_recommended_yaml(strategy_col: str, output_name: str) -> Path:
    data = yaml.safe_load(BASELINE_YAML.read_text(encoding="utf-8"))
    strat_map = dict(zip(all_products_features["product_name"], all_products_features[strategy_col]))

    n_composed = 0
    for product_name, product_cfg in data["models"][0]["bound_strategy"].items():
        if product_name in strat_map:
            product_cfg["type"] = strat_map[product_name]
            if strat_map[product_name] == "composed":
                n_composed += 1

    out_path = REPO_ROOT / "all_product_yamls" / output_name
    out_path.write_text(yaml.dump(data, sort_keys=False), encoding="utf-8")
    print(f"Écrit {out_path} — {n_composed}/{len(strat_map)} produits en composed")
    return out_path


tree_yaml_path = write_recommended_yaml("strategy_tree", "combo_recommended__tree.yaml")
rbf_yaml_path = write_recommended_yaml("strategy_rbf", "combo_recommended__rbf.yaml")


### 10b. Lancer la vérification sur le vrai solveur

Ces deux configs combinent **en un seul run** toutes les recommandations
(contrairement aux combos `all_product_yamls` d'origine qui ne testent qu'UN
produit à la fois). Depuis la racine du repo (environnement `certif`, WSL,
MOSEK configuré) :


In [ ]:
print(f'''
cd {REPO_ROOT}
python src/certification_problem.py blob_4x10 recommended_tree --config all_product_yamls/combo_recommended__tree.yaml
python src/certification_problem.py blob_4x10 recommended_rbf  --config all_product_yamls/combo_recommended__rbf.yaml
''')


### 10c. Comparer les résultats une fois les runs terminés

Une fois ces deux runs terminés, chargez leurs `results.csv` (dans
`results/benchmark/blob_nn_4x10-1/...recommended_tree.../` et
`...recommended_rbf.../`) et comparez `certification_rate` / `optimal_value`
moyen contre le baseline (`combo_0000`) et contre "tout en composed" (si vous
l'avez testé), en réutilisant les fonctions `load_limited_csv` /
`certification_rate` du notebook `combo_summary_fixed.ipynb`.


In [ ]:
# Exemple, une fois les dossiers de résultats connus :
# RECOMMENDED_TREE_RESULTS = Path("results/benchmark/blob_nn_4x10-1/<dossier>_recommended_tree/results.csv")
# RECOMMENDED_RBF_RESULTS  = Path("results/benchmark/blob_nn_4x10-1/<dossier>_recommended_rbf/results.csv")
#
# for label, path in [("recommended_tree", RECOMMENDED_TREE_RESULTS), ("recommended_rbf", RECOMMENDED_RBF_RESULTS)]:
#     res = pd.read_csv(path)
#     cert_rate = (res["optimal_value"] > 0).mean()
#     print(f"{label}: certification_rate={cert_rate:.3f}, optimal_value_mean={res['optimal_value'].mean():.3f}")


## 11. Décision `composed` vs `one_variable` en code Python pur (aucun fichier externe)

Au lieu de charger un modèle `.joblib` à chaque appel, on **transforme
directement l'arbre entraîné en code Python** (une cascade de `if/else` sur
les seuils appris) : la décision se prend alors **instantanément, en mémoire,
dès que `LB`/`UB` sont calculés pendant l'exécution du solveur** — sans
dépendance à `sklearn`, `joblib`, ni à un fichier modèle à charger/déployer.

On régénère un arbre de régression sur `gain`, avec les features
`LB_neuron1, UB_neuron1, LB_neuron2, UB_neuron2, l, k` (toutes disponibles au
moment où le produit croisé est traité), puis on **génère le code source**
de la fonction de décision, qu'on écrit directement dans
`bound_type_predictor.py`.


In [ ]:
from sklearn.tree import DecisionTreeRegressor

PROD_FEATURES = ["LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2", "l", "k"]

X_prod = tree_df[PROD_FEATURES].fillna(tree_df[PROD_FEATURES].median())
y_prod_gain = tree_df["gain"]

# Même profondeur que la section 7, pour rester lisible et éviter le sur-apprentissage.
prod_tree_reg = DecisionTreeRegressor(max_depth=MAX_DEPTH_REG, min_samples_leaf=10, random_state=0)
prod_tree_reg.fit(X_prod, y_prod_gain)

print(f"Arbre entraîné sur {len(X_prod)} points, profondeur {prod_tree_reg.get_depth()}, "
      f"{prod_tree_reg.get_n_leaves()} feuilles.")


### 11a. Générateur : arbre sklearn → fonction Python autonome

In [ ]:
def tree_to_python_function(tree, feature_names, func_name="predict_gain_tree"):
    """Convertit un DecisionTreeRegressor/Classifier en code source Python
    autonome (if/else), sans dépendance à sklearn à l'exécution."""
    tree_ = tree.tree_
    lines = [f"def {func_name}({', '.join(feature_names)}):"]

    def recurse(node, depth):
        indent = "    " * (depth + 1)
        if tree_.feature[node] != -2:  # noeud interne (pas une feuille)
            name = feature_names[tree_.feature[node]]
            threshold = tree_.threshold[node]
            lines.append(f"{indent}if {name} <= {threshold:.6f}:")
            recurse(tree_.children_left[node], depth + 1)
            lines.append(f"{indent}else:")
            recurse(tree_.children_right[node], depth + 1)
        else:  # feuille
            value = float(tree_.value[node][0][0])
            lines.append(f"{indent}return {value:.6f}")

    recurse(0, 0)
    return "\n".join(lines)


tree_function_code = tree_to_python_function(prod_tree_reg, PROD_FEATURES, func_name="predict_gain_tree")
print(tree_function_code)


### 11b. Écrire `bound_type_predictor.py` (aucune dépendance externe)

In [ ]:
predictor_module_code = f'''"""Décision \'composed\' vs \'one_variable\' pour un produit croisé
z_{{l,u}} * z_{{k,j}}, générée automatiquement à partir d'un arbre de décision
entraîné hors ligne (notebook combo_visualisation, section 11).

Ce fichier ne dépend d'AUCUNE bibliothèque externe (ni sklearn, ni joblib) :
c'est du pur Python, donc la décision se prend instantanément, en mémoire,
au moment où LB/UB sont calculés pendant l'exécution du solveur.

⚠️ Régénérez ce fichier (notebook combo_visualisation, section 11) si le
réseau, l'epsilon, ou les cuts changent — les seuils ci-dessous sont
spécifiques aux données sur lesquelles l'arbre a été entraîné.
"""

from __future__ import annotations


{tree_function_code}


def predict_bound_type(
    l: int, u_idx: int, k: int, j_idx: int,
    LB_neuron1: float, UB_neuron1: float, LB_neuron2: float, UB_neuron2: float,
    gain_threshold: float = 0.0,
) -> str:
    """Décide \'composed\' ou \'one_variable\' pour un produit croisé, dès que ses
    bornes LB/UB sont connues. Pur Python, pas d'I/O, pas de dépendance.

    Args:
        l, k: indices de couche des 2 neurones du produit.
        u_idx, j_idx: indices de neurone (non utilisés par l'arbre actuel,
            gardés pour signature/logging cohérents avec le reste du pipeline).
        LB_neuron1, UB_neuron1, LB_neuron2, UB_neuron2: bornes pré-activation
            des 2 neurones (ex: self.L[l][u_idx], self.U[l][u_idx], ...).
        gain_threshold: seuil de gain prédit au-delà duquel \'composed\' est
            recommandé.

    Returns:
        "composed" ou "one_variable"
    """
    predicted_gain = predict_gain_tree(LB_neuron1, UB_neuron1, LB_neuron2, UB_neuron2, l, k)
    return "composed" if predicted_gain > gain_threshold else "one_variable"
'''

with open("bound_type_predictor.py", "w", encoding="utf-8") as f:
    f.write(predictor_module_code)

print("Écrit : bound_type_predictor.py (autonome, sans dépendance)")


### 11c. Vérification rapide

On recharge le fichier généré et on vérifie qu'il donne les mêmes prédictions
que l'arbre `sklearn` d'origine, sur quelques points au hasard.


In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("bound_type_predictor", "bound_type_predictor.py")
generated_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(generated_module)

sample = X_prod.sample(min(20, len(X_prod)), random_state=0)
sklearn_preds = prod_tree_reg.predict(sample)
generated_preds = [
    generated_module.predict_gain_tree(*row) for row in sample[PROD_FEATURES].itertuples(index=False)
]

max_diff = max(abs(a - b) for a, b in zip(sklearn_preds, generated_preds))
print(f"Écart max entre sklearn et le code généré : {max_diff:.2e} (doit être ~0)")


### 11d. Intégration dans le pipeline

Copiez le fichier `bound_type_predictor.py` généré (cellule 11b) dans
`src/fastsdp_tools/` de votre repo (il remplace la version précédente à base
de `.joblib`/`sklearn` — l'interface `predict_bound_type(...)` reste
identique, donc **rien à changer côté `variables_call.py`**).

```python
from fastsdp_tools.bound_type_predictor import predict_bound_type

bound_type = predict_bound_type(
    l=layer_prev, u_idx=neuron_prev, k=layer_next, j_idx=neuron_next,
    LB_neuron1=self.L[layer_prev][neuron_prev], UB_neuron1=self.U[layer_prev][neuron_prev],
    LB_neuron2=self.L[layer_next][neuron_next], UB_neuron2=self.U[layer_next][neuron_next],
)
```

Aucun fichier `.joblib` à générer, copier ou charger : la décision est prise
par une fonction Python pure, disponible dès l'import du module.

⚠️ Toujours spécifique au réseau/epsilon/cuts d'entraînement — régénérez ce
fichier (11a-11b) si vous changez de config.


## 12. Décision par point de données réel (pas une moyenne globale)

La section 10 utilisait les bornes LB/UB **moyennées sur tout le baseline**
pour recommander une config unique appliquée à toutes les images. Ici, on
fait la décision **pour UN point de données précis** (`data_index` donné),
avec SES propres bornes LB/UB — exactement ce que fait le hook dans
`variables_call.py` pendant un vrai run, mais rejouable ici pour inspection
avant de lancer quoi que ce soit.

Pour chacun des 330 produits du réseau, on regarde les bornes LB/UB de ses 2
neurones **pour ce data_index précis**, on appelle `predict_bound_type(...)`
(le module généré en section 11), et on liste les produits qui deviennent
`composed`.


In [ ]:
import yaml
from pathlib import Path

# === CONFIGURATION ===
REPO_ROOT = Path("FullSDPCertification")  # clone local de votre repo
BASELINE_YAML = REPO_ROOT / "all_product_yamls" / "combo_0000__baseline__all_one_variable.yaml"
BASELINE_STABLE_CSV = Path("results/benchmark/blob_nn_4x10-1/<dossier_combo_0000>/stable_actives_study.csv")

DATA_INDEX_TO_INSPECT = 0  # changez pour n'importe quel data_index testé par le baseline


In [ ]:
# === 1. Charger les 330 produits du réseau ===
baseline_yaml_data = yaml.safe_load(BASELINE_YAML.read_text(encoding="utf-8"))
bound_strategy_template = baseline_yaml_data["models"][0]["bound_strategy"]

products = []
for product_name, product_cfg in bound_strategy_template.items():
    l, u, k, j = product_cfg["key"]
    products.append({"product_name": product_name, "l": l, "u_idx": u, "k": k, "j_idx": j})

products_df = pd.DataFrame(products)
print(f"{len(products_df)} produits chargés")


In [ ]:
# === 2. Bornes LB/UB réelles POUR CE data_index précis (pas une moyenne) ===
baseline_stable_df = pd.read_csv(BASELINE_STABLE_CSV)
row = baseline_stable_df[baseline_stable_df["data_index"] == DATA_INDEX_TO_INSPECT]
if row.empty:
    raise ValueError(f"data_index={DATA_INDEX_TO_INSPECT} introuvable dans {BASELINE_STABLE_CSV}")
row = row.iloc[0]

def bounds_for(layer: int, neuron: int) -> tuple[float, float]:
    lb_col = f"LB_Layer_{layer}_Neuron_{neuron}"
    ub_col = f"UB_Layer_{layer}_Neuron_{neuron}"
    return float(row[lb_col]), float(row[ub_col])


print(f"Point de données inspecté : data_index={DATA_INDEX_TO_INSPECT}, label={row.get('label')}")


In [ ]:
# === 3. Décider composed/one_variable pour chaque produit, avec CES bornes précises ===
from bound_type_predictor import predict_bound_type, predict_gain_tree

records = []
for p in products_df.itertuples():
    lb1, ub1 = bounds_for(p.l, p.u_idx)
    lb2, ub2 = bounds_for(p.k, p.j_idx)
    gain = predict_gain_tree(lb1, ub1, lb2, ub2, p.l, p.k)
    bound_type = "composed" if gain > 0 else "one_variable"
    records.append({
        "product_name": p.product_name, "l": p.l, "u_idx": p.u_idx, "k": p.k, "j_idx": p.j_idx,
        "LB_neuron1": lb1, "UB_neuron1": ub1, "LB_neuron2": lb2, "UB_neuron2": ub2,
        "predicted_gain": gain, "bound_type": bound_type,
    })

decisions_df = pd.DataFrame(records).sort_values("predicted_gain", ascending=False)
n_composed = (decisions_df["bound_type"] == "composed").sum()
print(f"data_index={DATA_INDEX_TO_INSPECT} : {n_composed}/{len(decisions_df)} produits recommandés en 'composed'")
decisions_df.head(20)


### 12a. Écrire la config YAML spécifique à ce point de données

In [ ]:
def write_yaml_for_datapoint(decisions_df: pd.DataFrame, output_name: str) -> Path:
    data = yaml.safe_load(BASELINE_YAML.read_text(encoding="utf-8"))
    strat_map = dict(zip(decisions_df["product_name"], decisions_df["bound_type"]))

    for product_name, product_cfg in data["models"][0]["bound_strategy"].items():
        if product_name in strat_map:
            product_cfg["type"] = strat_map[product_name]

    out_path = REPO_ROOT / "all_product_yamls" / output_name
    out_path.write_text(yaml.dump(data, sort_keys=False), encoding="utf-8")
    print(f"Écrit {out_path}")
    return out_path


write_yaml_for_datapoint(decisions_df, f"combo_recommended__datapoint_{DATA_INDEX_TO_INSPECT}.yaml")


### 12b. Note importante

Cette config par point de données est utile pour **inspecter/déboguer** ce
que le modèle recommande sur un cas précis. Mais en production, ce n'est pas
comme ça que ça doit tourner : les bornes L/U ne sont connues **qu'au moment
où le solveur les calcule**, image par image — c'est pour ça que le vrai hook
est dans `variables_call.py::add_z_quad_active_neuron` (`mccormick_type=
"auto"`), qui appelle `predict_bound_type(...)` **à la volée pendant la
résolution**, avec les bornes du point en cours, sans jamais avoir besoin de
pré-générer un YAML par image.
